In [ ]:
import torch
from latent_models.latent_utils import get_latent_model

class Args: pass
args = Args()
args.enc_dec_model          = "facebook/bart-base"
args.num_encoder_latents    = 1
args.num_decoder_latents    = 1
args.dim_ae                 = 128
args.num_layers             = 2
args.l2_normalize_latents   = False
args.lm_mode                = "freeze"


lm, tokenizer, config = get_latent_model(args)  

ckpt = torch.load("./model.pt", map_location="cpu")
lm.load_state_dict(ckpt["model"], strict=False)

lm.eval()

Some weights of BARTForConditionalGenerationLatent were not initialized from the model checkpoint at facebook/bart-base and are newly initialized: ['perceiver_ae.perceiver_encoder.layers.1.0.to_out.0.weight', 'perceiver_ae.perceiver_encoder.layers.1.0.query_norm.gamma', 'perceiver_ae.perceiver_decoder.input_proj.weight', 'perceiver_ae.perceiver_decoder.input_proj.bias', 'perceiver_ae.perceiver_decoder.final_norm.weight', 'perceiver_ae.perceiver_encoder.layers.1.1.1.weight', 'perceiver_ae.perceiver_encoder.layers.0.1.0.bias', 'perceiver_ae.perceiver_decoder.layers.0.0.to_q.weight', 'perceiver_ae.perceiver_encoder.layers.0.1.0.weight', 'perceiver_ae.perceiver_decoder.final_norm.bias', 'perceiver_ae.perceiver_encoder.layers.0.0.to_kv.weight', 'perceiver_ae.perceiver_encoder.layers.0.0.to_out.0.weight', 'perceiver_ae.perceiver_decoder.layers.0.1.1.weight', 'perceiver_ae.perceiver_encoder.layers.0.0.key_norm.gamma', 'perceiver_ae.perceiver_decoder.layers.0.1.4.bias', 'perceiver_ae.perceiver

Trainable: perceiver_ae.perceiver_encoder.latents
Trainable: perceiver_ae.perceiver_encoder.pos_emb.emb.weight
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.norm.weight
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.norm.bias
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.norm_latents.weight
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.norm_latents.bias
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.query_norm.gamma
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.key_norm.gamma
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.to_q.weight
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.latent_to_kv.weight
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.to_kv.weight
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.to_out.0.weight
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.to_out.0.bias
Trainable: perceiver_ae.perceiver_encoder.layers.0.1.0.weight
Trainable: perceiver_ae.perceiver_encoder.layers.0.1.0.bias
Trainable: percei

In [6]:
import re

with open("./output.txt") as f:
    text = f.read()

def extract_blocks(text):
    blocks = []
    idx = 0
    while True:
        pos = text.find("responses", idx)
        if pos == -1:
            break
        start = text.find("[", pos)
        if start == -1:
            break
        depth = 0
        for i, ch in enumerate(text[start:], start):
            if ch == "[":
                depth += 1
            elif ch == "]":
                depth -= 1
                if depth == 0:
                    blocks.append(text[start:i+1])
                    idx = i + 1
                    break
        else:
            break
    return blocks

blocks = extract_blocks(text)
responses_lists = []
for blk in blocks:
    try:
        responses = eval(blk, {"tensor": lambda *args, **kwargs: None})
        responses_lists.append(responses)
    except:
        pass

from pprint import pprint
pprint(responses_lists)


[[('radio wave communications',
   [-3.650303602218628, -4.025141716003418, -1.7355194091796875],
   None,
   0.0),
  ('wireless data transfer',
   [-1.789168119430542, -1.6794599294662476, -0.6588299870491028],
   None,
   0.0),
  ('portable communications',
   [-3.908777952194214, -2.921856164932251],
   None,
   0.0),
  ('communication', [-3.1746928691864014], None, 0.0),
  ('communication', [-3.1746928691864014], None, 0.0),
  ('near field communication',
   [-4.673958778381348, -1.017874002456665, -0.503303050994873],
   None,
   0.0),
  ('phones', [-4.317170143127441], None, 0.0),
  ('mobile phones', [-1.1736953258514404, -0.434695839881897], None, 0.0),
  ('communication between devices',
   [-3.1746928691864014, -1.8701362609863281, -1.5504412651062012],
   None,
   0.0),
  ('bluetooth', [-4.2521562576293945], None, 0.0)],
 [('the address', [-0.7398093938827515, -3.4815847873687744], None, 0.0),
  ('a letter head',
   [-2.325260639190674, -1.8908360004425049, -2.750032901763916

In [7]:
texts_only = [[resp[0] for resp in resp_list] for resp_list in responses_lists]

from pprint import pprint
pprint(texts_only)


[['radio wave communications',
  'wireless data transfer',
  'portable communications',
  'communication',
  'communication',
  'near field communication',
  'phones',
  'mobile phones',
  'communication between devices',
  'bluetooth'],
 ['the address',
  'a letter head',
  'space',
  'the top left hand corner',
  'salutation',
  'the letter head',
  'its height',
  'the capital',
  'the tail',
  'the space between the capital letter and the lower case letter'],
 ['count of russia and les mizhi',
  'lord of the rings',
  "Ivan Turgenev's Fathers and Sons",
  'war and peace',
  'war and peace',
  'War and Peace',
  'war and peace',
  'the 3rd book of the "Gogol" trilogy "The Inspector General"',
  'war and peace',
  'war and peace'],
 ['Joseph Goebbels',
  'joachim stiehl',
  'Josef Goebbels',
  'joseph goebbels',
  'Goebbels',
  'josef goebbels',
  'martin bormann',
  'Adolf Hitler',
  'goebbels',
  'joachim stahlecker'],
 ['new york',
  'dublin',
  'paris',
  'NEW YORK',
  'london',


In [8]:
# save the tensors to a file
with open("tensors.txt", "w") as f:
    for group in texts_only:
        for text in group:
            print(text)
            inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
            enc_opts = lm.get_encoder()(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask)
            latents  = lm.get_diffusion_latent(enc_opts, inputs.attention_mask)
            print("Raw latent tensor:\n", latents)
            print(latents.shape)
            f.write(str(text) + "\n")
            f.write(str(latents.tolist()) + "\n")
            f.write("\n")



radio wave communications
Raw latent tensor:
 tensor([[[-1.0104, -1.0151,  1.4566,  0.6206,  0.6528, -0.0065, -0.0252,
          -0.0664, -0.8623, -0.2798,  1.1484, -1.0327,  1.3266, -1.2295,
           0.1140, -0.7283,  1.6096, -0.2701, -1.6137,  0.9710,  0.5593,
           0.2365, -0.3430, -0.2496,  1.6476,  0.7779, -0.0734, -1.7190,
           0.0869, -1.0447, -0.1886, -0.0110,  1.5642,  1.0332,  1.8097,
          -0.0881,  1.7933, -0.3100, -0.3581,  1.4792,  0.0505, -1.1057,
          -1.1895,  0.0330,  0.9297, -0.3150, -1.0140,  0.6823, -0.6671,
          -0.0758,  0.5258, -1.4147, -0.8374,  0.5255,  0.9423,  0.2327,
           0.3649, -0.6804,  0.8309, -2.1598,  1.1388, -0.3505,  0.1270,
          -0.2997,  0.8260,  1.0487, -0.1368, -0.0178, -1.3270, -0.2375,
           0.8572, -0.6393,  0.6097, -1.5434, -0.4138, -2.2167, -0.1264,
           1.5220, -1.0014,  1.7276, -0.1471, -0.7349, -1.0853,  0.8208,
          -0.6359,  0.7498,  1.6177,  0.1574, -0.0427,  0.1727, -1.2043,
     

In [4]:
import torch
from latent_models.latent_utils import get_latent_model

class Args:
    enc_dec_model = "facebook/bart-base"
    lm_mode       = "freeze"

args = Args()
lm, tokenizer, config = get_latent_model(args)

ckpt = torch.load("./model.pt", map_location="cpu")
lm.load_state_dict(ckpt["model"], strict=False)
lm.eval()

inputs  = tokenizer("It didn't rained today", return_tensors="pt", truncation=True, padding=True)
enc_out = lm.get_encoder()(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask)
latents = lm.get_diffusion_latent(enc_out, inputs.attention_mask)

print(latents)
print(latents.shape)

with open("tensors_bart.txt", "w") as f:
    for group in texts_only:
        for text in group:
            inputs  = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
            enc_out = lm.get_encoder()(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask)
            latents = lm.get_diffusion_latent(enc_out, inputs.attention_mask)
            f.write(text + "\n")
            f.write(str(latents.tolist()) + "\n\n")

Some weights of BARTForConditionalGenerationLatent were not initialized from the model checkpoint at facebook/bart-base and are newly initialized: ['perceiver_ae.perceiver_encoder.layers.2.0.to_out.0.weight', 'perceiver_ae.perceiver_encoder.layers.1.0.to_out.0.weight', 'perceiver_ae.perceiver_decoder.layers.3.1.4.weight', 'perceiver_ae.perceiver_encoder.layers.2.1.4.bias', 'perceiver_ae.perceiver_encoder.layers.5.1.1.weight', 'perceiver_ae.perceiver_encoder.layers.1.0.query_norm.gamma', 'perceiver_ae.perceiver_decoder.input_proj.weight', 'perceiver_ae.perceiver_decoder.input_proj.bias', 'perceiver_ae.perceiver_decoder.layers.4.0.norm.bias', 'perceiver_ae.perceiver_decoder.final_norm.weight', 'perceiver_ae.perceiver_encoder.layers.3.0.key_norm.gamma', 'perceiver_ae.perceiver_encoder.layers.4.0.key_norm.gamma', 'perceiver_ae.perceiver_decoder.layers.5.0.query_norm.gamma', 'perceiver_ae.perceiver_encoder.layers.1.1.1.weight', 'perceiver_ae.perceiver_decoder.layers.2.0.to_q.weight', 'perce

Trainable: perceiver_ae.perceiver_encoder.latents
Trainable: perceiver_ae.perceiver_encoder.pos_emb.emb.weight
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.norm.weight
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.norm.bias
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.norm_latents.weight
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.norm_latents.bias
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.query_norm.gamma
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.key_norm.gamma
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.to_q.weight
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.to_kv.weight
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.to_out.0.weight
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.to_out.0.bias
Trainable: perceiver_ae.perceiver_encoder.layers.0.1.0.weight
Trainable: perceiver_ae.perceiver_encoder.layers.0.1.0.bias
Trainable: perceiver_ae.perceiver_encoder.layers.0.1.1.weight
Trainable: perceiver_ae.perc